# Track1 FabricIQ: 원본 14개 CSV 분석 및 Ontology 완료

이 Notebook은 원본 워크숍의 14개 CSV를 검증해 Lakehouse Delta 테이블로 적재하고, Q1~Q5 지표와 원본→표준 컬럼→Ontology 바인딩을 검증합니다. 마지막 단계에서 표준화 backing table을 만들고 Ontology 정의를 갱신합니다.

## 실행 전 확인
- 이 Notebook을 실행할 Fabric Workspace의 Lakehouse를 기본 Lakehouse로 연결하세요.
- 원본 CSV 14개를 해당 Lakehouse의 `Files/` 루트에 올려야 합니다. 파일은 저장소의 `track1/data/`에 있습니다.
- 데이터 해시와 행 수가 고정된 원본 버전과 일치해야 합니다. 불일치하면 실행을 중단합니다.
- `workshop_` 테이블을 이미 모두 적재했다면 적재를 건너뛰고 다시 사용합니다. 일부 테이블만 있으면 임의로 덮어쓰지 않고 중단합니다.
- 마지막 단계는 `IQ_Workshop_Track1_Ontology` 전체 정의를 저장소 생성물로 갱신합니다. 기존 Ontology에 수동으로 추가한 정의는 보존되지 않으므로 실행 전에 백업하세요.


## 1. 데이터 검증 및 적재 함수

먼저 Python 표준 라이브러리로 CSV의 SHA-256, 열 이름, 행 수를 확인합니다. 적재할 때는 원본 상태를 보존하기 위해 모든 열을 문자열로 읽고 Delta 테이블에 씁니다. 기존 테이블을 덮어쓰지 않으며 저장 후 행 수도 재확인합니다.

In [ ]:
"""기본 Lakehouse를 연결한 Fabric Notebook에서 실행하는 원본 CSV 적재 코드."""

import csv
import hashlib
import io
from pathlib import Path

DATA_ROOT = Path("/lakehouse/default/Files")
SPARK_DATA_ROOT = "Files"
# 원본 CSV별 예상 행 수와 SHA-256입니다. 데이터가 바뀌면 적재를 중단합니다.
ROWS = {
    "channels": 4, "customers": 1200, "products": 1200, "campaigns": 1000,
    "promotions": 1000, "orders": 2003, "order_items": 4024, "payments": 2004,
    "shipments": 2004, "returns": 1000, "inventory_snapshots": 2096,
    "order_promotions": 1910, "campaign_attribution": 2250, "support_tickets": 1314,
}
SHA256 = {
    "channels": "2ce04ff27fd0662103f205e0d760dcf6da19502498ced244f729a9eaf23c576e",
    "customers": "ad5df1bf49b8cb3c3f11365656e328938f159eccab83b605aba481c001ac29ac",
    "products": "3adb8f5ee6236bcef208a1af6eeea2ec837e7708284c1bc685a301035e314c97",
    "campaigns": "8b5eb8156bcaa426e45e525338d2edc291382f0465c4421ee87b378d64ee7a4a",
    "promotions": "9f42a16b76c66ad65c8e4e78a89bf52dafc309c42d2497fa35a362a42b8e17c7",
    "orders": "87f199e0876f4ef1137107036ed563ab3d06829fc7c48d7d0a72ec626d0c0348",
    "order_items": "50aae9fd67062d3c46e7de935dc956d2c4e16577975d414df676fe9906e742c5",
    "payments": "6c6bae26fc5c0740900379b10a730724211a550c085398664d5686cf4fe21886",
    "shipments": "fed38bf19d58b61d69cf2c799505f5de1919f7a2154818cd5d760d9f59b1051e",
    "returns": "0c057f1cc79fa432b5d711b825c0e3509e5711f7da558e1210d8822612ef7d9b",
    "inventory_snapshots": "b7d1e7a0aeb8cdd12d9589a8ef81497a68c1b3b17b7aa3fd54e529bf7d6bdfec",
    "order_promotions": "69a00ddf80d47b07e41ea9d6ab4c4fb7db21a27adf4f4528a88528b678513f25",
    "campaign_attribution": "61f104752acaecd70cda95c85df07e6d6197fee413e4788885600778493b3283",
    "support_tickets": "2007e010ad4d7d50a1f79ce25d22d98780cee9a82ffa1d24434a1aaafa244fe0",
}


def verify_source_files():
    """CSV의 원본 해시, 열 머리글, 행 수를 확인하고 열 이름을 반환합니다."""
    headers = {}
    for name, expected in ROWS.items():
        path = DATA_ROOT / f"{name}.csv"
        if not path.is_file():
            raise FileNotFoundError(path)
        content = path.read_bytes()
        if hashlib.sha256(content).hexdigest() != SHA256[name]:
            raise ValueError(f"{name}: SHA-256 mismatch")
        with io.StringIO(content.decode("utf-8-sig"), newline="") as stream:
            reader = csv.reader(stream, strict=True)
            columns = next(reader, None)
            if not columns or len(columns) != len(set(columns)):
                raise ValueError(f"{name}: invalid or duplicate CSV headers")
            actual_rows = sum(1 for _ in reader)
        if actual_rows != expected:
            raise ValueError(f"{name}: expected {expected} rows, got {actual_rows}")
        headers[name] = columns
        print(f"{name}.csv: {actual_rows}행, 원본 해시 일치")
    return headers


def load(spark):
    # 기존 테이블은 덮어쓰지 않습니다. 재적재 전에는 테이블을 직접 검토하세요.
    existing = [f"workshop_{name}" for name in ROWS
                if spark.catalog.tableExists(f"workshop_{name}")]
    if existing:
        raise RuntimeError(f"Existing tables; review before loading: {existing}")
    headers = verify_source_files()
    frames = {}
    for name, expected in ROWS.items():
        # Spark에서 Lakehouse Files의 CSV를 읽고 CSV 열 이름이 같은지 확인합니다.
        frame = (spark.read.option("header", True).option("inferSchema", False)
                 .option("mode", "FAILFAST").option("multiLine", True)
                 .csv(f"{SPARK_DATA_ROOT}/{name}.csv"))
        if frame.columns != headers[name]:
            raise ValueError(f"{name}: CSV column mismatch")
        actual = frame.count()
        if actual != expected:
            raise ValueError(f"{name}: expected {expected} rows, got {actual}")
        frames[name] = frame
    created = []
    try:
        for name, frame in frames.items():
            table = f"workshop_{name}"
            # 원본 컬럼을 문자열로 보존해 의도된 데이터 품질 사례를 변경하지 않습니다.
            frame.write.format("delta").mode("errorifexists").saveAsTable(table)
            created.append(table)
            if spark.table(table).count() != ROWS[name]:
                raise RuntimeError(f"{table}: saved row count mismatch")
            print(f"{table}: {ROWS[name]} rows")
    except Exception as exc:
        raise RuntimeError(f"Load failed; inspect these created tables before retry: {created}") from exc

## 2. CSV와 Lakehouse 테이블 상태 확인

CSV 14개의 체크섬과 행 수를 먼저 검증합니다. 테이블이 전혀 없으면 적재하고, 14개가 이미 있으면 기존 데이터를 확인만 합니다. 일부 테이블만 있는 경우에는 수동 검토가 필요하므로 중단합니다.

In [ ]:
# 원본 CSV 파일이 모두 예상된 버전인지 먼저 확인합니다.
source_headers = verify_source_files()

existing_tables = [
    name for name in ROWS
    if spark.catalog.tableExists(f"workshop_{name}")
]

if not existing_tables:
    # 아직 테이블이 없을 때만 14개 Delta 테이블을 새로 만듭니다.
    load(spark)
elif len(existing_tables) == len(ROWS):
    # 이미 전체가 있으면 재적재하지 않고 각 테이블의 행 수를 확인합니다.
    for name, expected_rows in ROWS.items():
        actual_rows = spark.table(f"workshop_{name}").count()
        if actual_rows != expected_rows:
            raise ValueError(
                f"workshop_{name}: 예상 {expected_rows}행, 실제 {actual_rows}행"
            )
        print(f"workshop_{name}: {actual_rows}행 확인")
else:
    raise RuntimeError(
        "일부 workshop_ 테이블만 있습니다. 기존 테이블을 확인한 뒤 직접 정리하세요: "
        + ", ".join(existing_tables)
    )


## 3. Q1~Q5 분석

각 쿼리는 `\n` 줄바꿈을 포함한 문자열로 만들어 Spark SQL에 전달합니다. 주문, 결제 시도, 반품, 문의, 캠페인 귀속은 서로 행 수가 다르므로 각 쿼리의 집계 단위를 확인하세요. 결과는 원본 데이터의 의도된 품질 이슈와 합성된 신호를 포함할 수 있으며 운영 의사결정의 근거로 사용하지 않습니다.

### Q1. 캠페인별 결제 승인

캠페인 귀속 주문을 중복 제거하고 주문별 결제 상태를 계산합니다. 승인율은 방문자 전환율이 아닙니다. 미확인 결제 상태는 별도로 셉니다.

In [ ]:
sql_q1 = (
    "-- Q1: 캠페인별 주문을 한 번씩 세고, 성공 결제가 있으면 승인으로 분류합니다.\n"
    "WITH payment_flags AS (\n"
    "  SELECT order_id,\n"
    "         MAX(CASE WHEN UPPER(TRIM(payment_status)) IN ('SUCCESS','RETRYSUCCESS') THEN 1 ELSE 0 END) AS approved,\n"
    "         MAX(CASE WHEN UPPER(TRIM(payment_status)) = 'FAILED' THEN 1 ELSE 0 END) AS failed\n"
    "  FROM workshop_payments GROUP BY order_id\n"
    "), attributed AS (\n"
    "  SELECT DISTINCT campaign_id, order_id FROM workshop_campaign_attribution\n"
    ")\n"
    "SELECT c.campaign_id, c.campaign_name, COUNT(*) AS order_count,\n"
    "       SUM(COALESCE(p.approved, 0)) AS approved_orders,\n"
    "       SUM(CASE WHEN COALESCE(p.approved, 0) = 0 AND p.failed = 1 THEN 1 ELSE 0 END) AS failed_orders,\n"
    "       SUM(CASE WHEN COALESCE(p.approved, 0) = 0 AND COALESCE(p.failed, 0) = 0 THEN 1 ELSE 0 END) AS unknown_orders,\n"
    "       ROUND(100.0 * SUM(COALESCE(p.approved, 0)) / COUNT(*), 2) AS approval_rate_pct\n"
    "FROM attributed a\n"
    "JOIN workshop_campaigns c ON c.campaign_id = a.campaign_id\n"
    "LEFT JOIN payment_flags p ON p.order_id = a.order_id\n"
    "WHERE EXISTS (SELECT 1 FROM workshop_orders o WHERE o.order_id = a.order_id)\n"
    "GROUP BY c.campaign_id, c.campaign_name ORDER BY c.campaign_id;"
)
result_q1 = spark.sql(sql_q1)
display(result_q1)

### Q2. 배송 지연과 반품·불만

배송 주문 단위로 지연 여부를 나누고 반품 주문 및 불만 티켓이 하나 이상 있는 주문의 비율을 계산합니다. 여러 티켓·반품이 있어도 주문은 한 번만 셉니다.

In [ ]:
sql_q2 = (
    "-- Q2: 반품·문의가 여러 건이어도 주문 단위로 한 번만 집계합니다.\n"
    "WITH shipment_flags AS (\n"
    "  SELECT order_id, MAX(CASE WHEN UPPER(TRIM(shipment_status)) = 'DELAYED' THEN 1 ELSE 0 END) AS delayed\n"
    "  FROM workshop_shipments GROUP BY order_id\n"
    "), returned AS (\n"
    "  SELECT DISTINCT order_id FROM workshop_returns\n"
    "), complaints AS (\n"
    "  SELECT DISTINCT order_id FROM workshop_support_tickets\n"
    "  WHERE UPPER(TRIM(ticket_type)) = 'COMPLAINT'\n"
    "), orders AS (\n"
    "  SELECT DISTINCT order_id FROM workshop_orders\n"
    ")\n"
    "SELECT s.delayed, COUNT(*) AS orders_count,\n"
    "       SUM(CASE WHEN r.order_id IS NOT NULL THEN 1 ELSE 0 END) AS returned_orders,\n"
    "       SUM(CASE WHEN t.order_id IS NOT NULL THEN 1 ELSE 0 END) AS complaint_orders,\n"
    "       ROUND(100.0 * SUM(CASE WHEN r.order_id IS NOT NULL THEN 1 ELSE 0 END) / COUNT(*), 2) AS return_rate_pct,\n"
    "       ROUND(100.0 * SUM(CASE WHEN t.order_id IS NOT NULL THEN 1 ELSE 0 END) / COUNT(*), 2) AS complaint_rate_pct\n"
    "FROM shipment_flags s JOIN orders o ON o.order_id = s.order_id\n"
    "LEFT JOIN returned r ON r.order_id = s.order_id\n"
    "LEFT JOIN complaints t ON t.order_id = s.order_id\n"
    "GROUP BY s.delayed ORDER BY s.delayed;"
)
result_q2 = spark.sql(sql_q2)
display(result_q2)

### Q3. 프로모션 유형과 재구매

프로모션 유형별 순액/총액 비율은 할인 효과의 대리 지표이지 원가 기반 마진이 아닙니다. 반복 고객은 전체 주문 이력에서 두 번 이상 주문한 고객입니다.

In [ ]:
sql_q3 = (
    "-- Q3: 순액/총액은 할인 효과의 대리 지표이며 원가 기반 마진이 아닙니다.\n"
    "WITH unique_orders AS (\n"
    "  SELECT DISTINCT order_id, customer_id, gross_amount, net_amount FROM workshop_orders\n"
    "), repeat_customers AS (\n"
    "  SELECT customer_id, CASE WHEN COUNT(DISTINCT order_id) >= 2 THEN 1 ELSE 0 END AS is_repeat\n"
    "  FROM unique_orders GROUP BY customer_id\n"
    "), promotion_orders AS (\n"
    "  SELECT DISTINCT op.order_id, p.promotion_type\n"
    "  FROM workshop_order_promotions op JOIN workshop_promotions p ON p.promotion_id = op.promotion_id\n"
    ")\n"
    "SELECT po.promotion_type, COUNT(*) AS orders_count,\n"
    "       ROUND(1.0 * SUM(CAST(o.net_amount AS DOUBLE)) / NULLIF(SUM(CAST(o.gross_amount AS DOUBLE)), 0), 4) AS net_to_gross,\n"
    "       ROUND(100.0 * AVG(rc.is_repeat), 2) AS repeat_customer_pct\n"
    "FROM promotion_orders po JOIN unique_orders o ON o.order_id = po.order_id\n"
    "JOIN repeat_customers rc ON rc.customer_id = o.customer_id\n"
    "GROUP BY po.promotion_type ORDER BY po.promotion_type;"
)
result_q3 = spark.sql(sql_q3)
display(result_q3)

### Q4. 품절 상품 포함 주문

한 번이라도 품절로 기록된 상품을 포함한 주문을 나눠 취소율과 주문당 문의 건수를 계산합니다. 재고 스냅샷 시점과 주문 시점이 연결되지 않아 인과관계를 뜻하지 않습니다.

In [ ]:
sql_q4 = (
    "-- Q4: 재고 스냅샷 시점과 주문 시점은 연결하지 않습니다.\n"
    "WITH stockout_products AS (\n"
    "  SELECT DISTINCT product_id FROM workshop_inventory_snapshots\n"
    "  WHERE CAST(on_hand_qty AS DOUBLE) - CAST(reserved_qty AS DOUBLE) <= 0\n"
    "), order_flags AS (\n"
    "  SELECT o.order_id,\n"
    "         MAX(CASE WHEN sp.product_id IS NOT NULL THEN 1 ELSE 0 END) AS has_stockout_product,\n"
    "         MAX(CASE WHEN UPPER(TRIM(o.order_status)) IN ('CANCELLED','CANCELED') THEN 1 ELSE 0 END) AS cancelled\n"
    "  FROM workshop_orders o JOIN workshop_order_items i ON i.order_id = o.order_id\n"
    "  LEFT JOIN stockout_products sp ON sp.product_id = i.product_id GROUP BY o.order_id\n"
    "), ticket_counts AS (\n"
    "  SELECT order_id, COUNT(*) AS ticket_count FROM workshop_support_tickets GROUP BY order_id\n"
    ")\n"
    "SELECT f.has_stockout_product, COUNT(*) AS orders_count,\n"
    "       ROUND(100.0 * AVG(f.cancelled), 2) AS cancellation_rate_pct,\n"
    "       ROUND(AVG(COALESCE(t.ticket_count, 0) * 1.0), 2) AS tickets_per_order\n"
    "FROM order_flags f LEFT JOIN ticket_counts t ON t.order_id = f.order_id\n"
    "GROUP BY f.has_stockout_product ORDER BY f.has_stockout_product;"
)
result_q4 = spark.sql(sql_q4)
display(result_q4)

### Q5. 채널·등급·반품 사유

반품 고객을 채널, 고객 등급, 반품 사유로 나눕니다. 반복 주문은 전체 이력의 2회 이상 주문으로 정의하며 반품 이후 재구매를 의미하지 않습니다.

In [ ]:
sql_q5 = (
    "-- Q5: 반복 주문은 전체 이력 기준이며 반품 이후 재구매를 뜻하지 않습니다.\n"
    "WITH unique_orders AS (\n"
    "  SELECT DISTINCT order_id, customer_id, channel_id FROM workshop_orders\n"
    "), repeat_customers AS (\n"
    "  SELECT customer_id, CASE WHEN COUNT(DISTINCT order_id) >= 2 THEN 1 ELSE 0 END AS is_repeat\n"
    "  FROM unique_orders GROUP BY customer_id\n"
    "), segments AS (\n"
    "  SELECT DISTINCT ch.channel_name, c.customer_tier, r.return_reason, o.customer_id\n"
    "  FROM workshop_returns r JOIN unique_orders o ON o.order_id = r.order_id AND o.customer_id = r.customer_id\n"
    "  JOIN workshop_customers c ON c.customer_id = o.customer_id\n"
    "  JOIN workshop_channels ch ON ch.channel_id = o.channel_id\n"
    ")\n"
    "SELECT s.channel_name, s.customer_tier, s.return_reason, COUNT(*) AS customers_count,\n"
    "       ROUND(100.0 * AVG(rc.is_repeat), 2) AS repeat_customer_pct\n"
    "FROM segments s JOIN repeat_customers rc ON rc.customer_id = s.customer_id\n"
    "GROUP BY s.channel_name, s.customer_tier, s.return_reason\n"
    "ORDER BY s.channel_name, s.customer_tier, s.return_reason;"
)
result_q5 = spark.sql(sql_q5)
display(result_q5)

## 완료 확인

다섯 결과가 모두 표시되면 분석 실행이 완료된 것입니다. 결과 행 수는 Q1 60개 캠페인, Q2 2개 배송 그룹, Q3 4개 프로모션 유형, Q4 2개 재고 그룹, Q5 최대 81개 세그먼트입니다. 값은 로컬 검증 결과와 비교하고, 다른 데이터 버전에서는 해시 검증 후에만 비교하세요.


In [ ]:
import hashlib
import json
from notebookutils import mssparkutils

# 각 SQL 결과의 전체 행을 로컬 기준 결과와 비교합니다.
spark.conf.set("spark.sql.jsonGenerator.ignoreNullFields", "false")
result_frames = {
    "Q1": result_q1, "Q2": result_q2, "Q3": result_q3,
    "Q4": result_q4, "Q5": result_q5,
}
expected_hashes = {'Q1': 'f8de6ad7c947a6665292ee3d23e672e4932ddd9a8657cf648aa8d25f27f968c1', 'Q2': '7fc974d4f080a170f895342db878cbe1388f343018fda89937b60015fa616da9', 'Q3': 'ccfbacc19accee975de6fabea9b8c3194b0211153bb9c77cbe70fbaf951110c1', 'Q4': '2dbd81a86b03beb07238cec804ee5b9c096931cd38fa28214b7be36acf804d8d', 'Q5': '195a5474c5ebc10af14ded686ad702f9c52dd46fba9d60edeb25ccc0d322eb6f'}
actual_hashes = {}
row_counts = {}
for question, frame in result_frames.items():
    rows = [json.loads(value) for value in frame.toJSON().collect()]
    row_counts[question] = len(rows)
    canonical = json.dumps(rows, ensure_ascii=False, sort_keys=True, separators=(",", ":"))
    actual_hashes[question] = hashlib.sha256(canonical.encode("utf-8")).hexdigest()

report = {
    "status": "passed" if actual_hashes == expected_hashes else "mismatch",
    "row_counts": row_counts,
    "matches_local": actual_hashes == expected_hashes,
    "actual_hashes": actual_hashes,
}
print(json.dumps(report, ensure_ascii=False, indent=2))
if report["status"] != "passed":
    raise AssertionError("Fabric Q1-Q5 results do not match the local reference")

# Q1-Q5가 기준 결과와 일치한 뒤 표준 backing tables와 Ontology를 생성/갱신합니다.
import importlib
import sys
import traceback

script_root = "/lakehouse/default/Files"
if script_root not in sys.path:
    sys.path.insert(0, script_root)
WORKSPACE_ID = "3ca3e1fd-c598-49dd-8603-532a14fa373f"
LAKEHOUSE_ID = "8cc1480e-2830-4fe0-b9a5-cca959408aa5"
try:
    import create_ontology_fabric_v2
    create_ontology_fabric_v2 = importlib.reload(create_ontology_fabric_v2)
    ontology_result = create_ontology_fabric_v2.run(
        spark,
        token=notebookutils.credentials.getToken("pbi"),
        workspace_id=WORKSPACE_ID,
        lakehouse_id=LAKEHOUSE_ID,
        display_name="IQ_Workshop_Track1_Ontology",
        apply=True,
        update_existing=True,
    )
except Exception as error:
    failure = {"queries": report, "ontology": {"mode": "failed", "error": str(error), "traceback": traceback.format_exc()}}
    print(json.dumps(failure, ensure_ascii=False, indent=2))
    mssparkutils.notebook.exit(json.dumps(failure, ensure_ascii=False))

print(json.dumps({"queries": report, "ontology": ontology_result}, ensure_ascii=False, indent=2))
if ontology_result.get("mode") != "apply" or ontology_result.get("generation") != 2:
    raise RuntimeError("Track1 Ontology was not verified as a generation-2 applied definition")
mssparkutils.notebook.exit(json.dumps({"queries": report, "ontology": ontology_result}, ensure_ascii=False))
